In [1]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import Imagenette, SVHN
import numpy as np

import logging

import importlib
import FL_sim
import models_to_train

importlib.reload(FL_sim)
importlib.reload(models_to_train)
from models_to_train import ResNetPLModel
from FL_sim import FLSimulator, save_grads_f_applied_on_grads


torch.set_float32_matmul_precision('high')

In [2]:
# dataset = [
#     Imagenette(
#         root='./data/Imagenet', split=s,
#         transform=transforms.Compose([
#             transforms.Resize(256),
#             transforms.CenterCrop(224),
#             transforms.ToTensor(),
#             transforms.Normalize(
#                 mean=[0.485, 0.456, 0.406],
#                 std=[0.229, 0.224, 0.225]
#             )
#         ])
#     ) for s in ['train', 'val']]


dataset = [
    SVHN(
        root='./data/SVHN', split=s,
        transform=transforms.Compose([
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4377, 0.4438, 0.4728],
                std=[0.1980, 0.2010, 0.1970]
            ),
        ])
    ) for s in ['train', 'test']]

In [3]:
# def test_model_training():
#     from pytorch_lightning import Trainer
#     model = ResNetPLModel(num_classes=10,
#         resnet_version='resnet18', lr=0.005)
#     training_dataloader = torch.utils.data.DataLoader(
#         dataset[0], batch_size=14000, shuffle=True,
#         num_workers=10, persistent_workers=True)
#     test_dataloader = torch.utils.data.DataLoader(
#         dataset[1], batch_size=14000,
#         num_workers=2, persistent_workers=True)
#     trainer = Trainer(max_epochs=10, accelerator='cuda', log_every_n_steps=len(training_dataloader)//2)
#     trainer.fit(model, training_dataloader, test_dataloader)
# # test_model_training()

In [4]:
# dataset = [torch.utils.data.Subset(d, list(range(50))) for d in dataset]
# for i in range(10):
#     for d in dataset:
#         d.dataset.labels[i]=i

In [5]:
def pre_send_process(single_model_grad_agent):

    return single_model_grad_agent


def server_rec_process(all_encoded_model_grad):
    return all_encoded_model_grad

def applied_on_grads_before_optim(fl_model,
    worker_id, curr_round, current_epoch, batch_idx, *args, **kwargs):
    pass

In [6]:
k = 2  # Number of workers

batch_size = 13000
# batch_size /= 50 # imagenet
# batch_size /= 3 # resnet 50
batch_size = int(batch_size)


model = ResNetPLModel(num_classes=10, resnet_version='resnet18', lr=0.01,
    logging_disabled=True, applied_on_grads_before_optim=applied_on_grads_before_optim)

sim = FLSimulator(
    num_agents=k, communication_rounds=3, client_epochs_per_round=30,
    batch_size=batch_size, dataset_train=dataset[0], dataset_test=dataset[1],
    aggregation_method='fedavg', non_iid_flag=False, pl_model=model,
    pre_send_process=pre_send_process, server_rec_process=server_rec_process
)

logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
sim.do_train_global_model_and_set_local_model(num_epochs=5)
sim.run_simulation()


round 1/3 --------------------
  - reporting global model metrics
         test loss: 2.135, test auc: 0.645
         train loss: 2.137, train auc: 0.654
     > training agent 1/2
         test loss: 1.250, test auc: 0.908
         train loss: 1.158, train auc: 0.925
     > training agent 2/2
         test loss: 1.332, test auc: 0.893
         train loss: 1.236, train auc: 0.911
Aggregating gradients using FedAvg...

round 2/3 --------------------
  - reporting global model metrics
         test loss: 1.513, test auc: 0.880
         train loss: 1.529, train auc: 0.878
     > training agent 1/2
         test loss: 0.953, test auc: 0.945
         train loss: 0.796, train auc: 0.963
     > training agent 2/2
         test loss: 0.921, test auc: 0.948
         train loss: 0.770, train auc: 0.965
Aggregating gradients using FedAvg...

round 3/3 --------------------
  - reporting global model metrics
         test loss: 1.283, test auc: 0.917
         train loss: 1.278, train auc: 0.918
   

In [7]:
# for i in range(0, 6):
#     model = ResNetPLModel(num_classes=10, resnet_version='resnet18', lr=0.005, logging_disabled=True,
#                           applied_on_grads_before_optim=save_grads_f_applied_on_grads)
#
#     # save_folder_path = f"experiments/exp_data/gradients_resnet/gradients_resnet_t{i}/"
#     save_folder_path = f"exp_dump//"
#     model.args_for_f_on_grad['save_folder_path'] = save_folder_path
#
#     model.load_state_dict(torch.load('experiments/exp_data/resnet18_svhn.pth', map_location='cpu'))
#     k = 2
#     batch_size = 13000
#     batch_size /= 6 # more batches as samples
#     batch_size = int(batch_size)
#
#     sim = FLSimulator(
#         num_agents=k, communication_rounds=2, client_epochs_per_round=30,
#         batch_size=batch_size, dataset_train=dataset[0], dataset_test=dataset[1],
#         aggregation_method='fedavg', non_iid_flag=False, pl_model=model,
#         pre_send_process=pre_send_process, server_rec_process=server_rec_process,
#     )
#
#     logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
#     sim.run_simulation()